In [1]:
import math
class Tensor:
    def __init__(self, data, _children=(), op=''):
        self.data = float(data)
        self.grad = 0.0
        self._prev = set(_children)
        self._backward = lambda:None
        self.op = op

    def __repr__(self):
        return f'Tensor(data={self.data}, grad={self.grad})'

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        out = Tensor(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        out = Tensor(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        
        out._backward = _backward

        return out

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        return self + (-other) # reuses add and neg

    def __rsub__(self, other): # Reverse Substraction
        other = other if isinstance(other, Tensor) else Tensor(other)

        return other + (-self)

    def __pow__(self, power):
        assert isinstance(power, (int, float))


        out = Tensor(self.data ** power, (self,), f'**{power}')

        def _backward():
            self.grad += (power * (self.data ** (power - 1)) ) * out.grad

        out._backward = _backward

        return out

    def __truediv__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        return self * (other ** -1)

    def __rtruediv__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        return other * (self ** -1)

    def exp(self):
        out = Tensor(math.exp(self.data), (self,), 'exp')

        def _backward():
            self.grad += out.data * out.grad

        out._backward = _backward

        return out

    def log(self):
        out = Tensor(math.log(self.data), (self,), 'log')

        def _backward():
            self.grad += (1 / self.data) * out.grad

        out._backward = _backward

        return out

    def abs(self):
        out = Tensor(abs(self.data), (self,), 'abs')

        def _backward():
            self.grad += (1 if self.data>= 0 else -1) * out.grad

        out._backward = _backward

        return out

    def tanh(self):
        out = Tensor(math.tanh(self.data), (self,), 'tanh')

        def _backward():
            self.grad += (1 - out.data ** 2) * out.grad

        out._backward = _backward

        return out

    def zero_grad(self):
        self.grad = 0.0
                
    def backward(self):

        topo = []
    
        visited = set()
    
        def build(v):
    
            if v not in visited:
    
                visited.add(v)
    
                for child in v._prev:
                    build(child)
    
                topo.append(v)
    
        build(self)
    
        self.grad = 1.0
    
        for node in reversed(topo):
            node._backward()

In [2]:
a = Tensor(2.0)
b = Tensor(3.0)

c = a * b
d = c + a

d.backward()

print("d =", d.data)
print("a.grad =", a.grad)
print("b.grad =", b.grad)

d = 8.0
a.grad = 4.0
b.grad = 2.0


In [3]:
x = Tensor(2)

y = ((x + 3) * 4 + 5) / 2

print(y.data)

y.backward()

print(x.grad)

12.5
2.0


In [4]:
x = Tensor(2)

y = ((x.exp() + 1).log()) * x

y.backward()

print(y.data)
print(x.grad)

4.253856022085945
3.8885221669987375


## Activations

### Sigmoid
Formula:

$$ \sigma(x) = \frac{1}{1+e^{-x}} $$

In [9]:
import math
def sigmoid(x):
    out = Tensor(1 / (1 + math.exp(-x.data)), (x,), 'sigmoid')

    def _backward():

        s = out.data
        x.grad += s * (1 - s) * out.grad

    out._backward = _backward

    return out

In [10]:
x = Tensor(0)

y = sigmoid(x)

y.backward()

print(y.data)
print(x.grad)

0.5
0.25


### Tanh
Formula:

$$ \tanh(x) = \frac{e^x-e^{-x}} {e^x+e^{-x}} $$

In [11]:
def tanh(x):
    out = Tensor(math.tanh(x.data), (x,), 'tanh')

    def _backward():
        t = math.tanh(x.data)
        x.grad += (1 - t**2) * out.grad

    out._backward = _backward

    return out

In [12]:
x = Tensor(0)

y = tanh(x)

y.backward()

print(y.data)
print(x.grad)

0.0
1.0


### ReLU
Formula:

$$ ReLU(x) = \max(0,x) $$


In [16]:
def relu(x):
    out = Tensor(max(0, x.data), (x,), 'relu')

    def _backward():
        x.grad += (1.0 if x.data > 0 else 0.0) * out.grad

    out._backward = _backward

    return out

In [18]:
x = Tensor(-5)
y = relu(x)
y.backward()
print(x.grad)

0.0


### Leaky ReLU
Formula:

$$ f(x) = \begin{cases} x & x>0\\ \alpha x & x\le0 \end{cases} $$

In [19]:
def leaky_relu(x, alpha=0.01):
    out = Tensor(x.data if x.data > 0 else alpha * x.data, (x,), 'leaky_rely')

    def _backward():
        x.grad += (1.0 if x.data > 0 else alpha) * out.grad

    out._backward = _backward

    return out

In [ ]:
x = Tensor(-5)
y = leaky_relu(x, alpha=0.01)
y.backward()
print(x.grad)
print(y)

0.01
Tensor(data=-0.05, grad=1.0)
1.0


### ELU
Formula:

$$ ELU(x)= \begin{cases} x & x>0\\ \alpha(e^x-1) & x\le0 \end{cases} $$
	​


In [26]:
def elu(x, alpha=1.0):
    out = Tensor(x.data if x.data > 0 else alpha * (math.exp(x.data) - 1), (x,), 'elu')

    def _backward():
        x.gard = (1.0 if x.data > 0 else alpha * math.exp(x.data)) * out.grad

    out._backward = _backward

    return out

In [28]:
x = Tensor(-5)
y = elu(x)
y.backward()
print(x.grad)
print(y)

0.0
Tensor(data=-0.9932620530009145, grad=1.0)


### SELU
Formula:

$$ SELU(x) = \lambda \begin{cases} x & x>0\\ \alpha(e^x-1) & x\le0 \end{cases} $$

	​


In [29]:
def selu(x):
    alpha = 1.6732632423543772
    scale = 1.0507009873554805

    out = Tensor(scale * (x.data if x.data > 0 else alpha * (math.exp(x.data) - 1)), (x,), 'selu')

    def _backward():
        x.grad += (scale * (1.0 if x.data > 0 else alpha * math.exp(x.data))) * out.grad

    out._backward = _backward

    return out

In [ ]:
x = Tensor(2)
y = relu(x)
z = y * y
z.backward()
print(x.grad)

4.0
